# SVM

---

> 线性分类器, 使异类样本点距离**超平面**的间距最大化

$$
\begin{align}
y_i (\omega \cdot x_i - b) \ge 1 \\
\text{where} \ y_i \in \{+1, \ -1\}
\end{align}
$$

为了找到最佳的权重 $\omega$ 和 $b$, 定义Hinge Loss $J$:
$$
J = \lambda ||\omega||^2 + \dfrac{1}{N} \sum_{i=1}^N \max(0, 1 - y_i(\omega \cdot x_i - b))
$$

其中 $\lambda$ 为系数, $||\omega||^2$ 为正则项。

In [1]:
import numpy as np

class SVN():
    def __init__(self, lr=1e-3, lambda_param=1e-2, num_iters=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.num_iters = num_iters
        self.w = None
        self.b = None
        
    def fit(self, X, y):
        """
        确定样本数和样本特征数
        对于y中的样本, 如果为负就赋为-1, 否则为1, 保证符合SVM求解特征
        """
        num_samples, num_features = X.shape
        y = np.where(y <= 0, -1, 1)
        
        """
        初始化参数w和b
        使用梯度下降法求解最优参数, 对于训练集中的每个样本: J = lambda * ||w||^2 + (1 - y_i (w * x_i - b))
            1. 假如分类正确, 即满足condition
                w = w - lr * dw = w - lr * (2 * lambda * w), 即只有正则项
                b = b - lr * db = b - lr * 0 = b, 没有后面的项b自然为0
            2. 假如分类错误, 即不满足condition
                w = w - lr * dw = w - lr * (2 * lambda * w - X[i] * y[i])
                b = b - lr * db = b - lr * y[i]
        """
        self.w = np.zeros(num_features)
        self.b = 0
        
        for _ in range(self.num_iters):
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(self.w, x_i) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y[idx]))
                    self.b -= self.lr * y[idx]
    
    """
    np.sign会根据传入的正/负值返回+1/-1
    """            
    def predict(self, X):
        out = np.dot(self.w, X) - self.b
        return np.sign(out)